In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import tqdm
import helpers

import environments
import environments.bicycle as bicycle

import controllers.stanley as stanley
import controllers.clothoids as clothoids
import controllers.purepursuit as purepursuit
import controllers.dqn as dqn

import metrics

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

In [ ]:
bicycle.create_rectangular_track_with_cross(
    center=(50.0, 50.0),
    length=80.0,
    width=40.0,
    turn_radius=8.0,
    lane_config=bicycle.lane_config_from_width(8.0, num_lanes=1),
)

bicycle.create_rectangular_track(
    center=(50.0, 50.0),
    length=80.0,
    width=40.0,
    turn_radius=8.0,
    lane_config=bicycle.lane_config_from_width(8.0, num_lanes=1),
)

In [ ]:
env = environments.bicycle.BicycleCarEnv(
    road_network=bicycle.RoadNetwork(roads=[
        bicycle.create_rectangular_track(
            center=(50.0, 50.0),
            length=80.0,
            width=40.0,
            turn_radius=8.0,
            lane_config=bicycle.lane_config_from_width(8.0, num_lanes=1),
        )
    ]),
    # road_network=bicycle.create_rectangular_track_with_cross(
    #     center=(50.0, 50.0),
    #     length=80.0,
    #     width=40.0,
    #     cross_width=4.5,
    #     turn_radius=8.0,
    #     lane_config=bicycle.lane_config_from_width(8.0, num_lanes=1),
    # ),
    render_mode="rgb_array",
    spawn=((50.0, 30.0), 0.0),
    goal=((10.0, 50.0), 2.0),
    obstacles=[
        bicycle.Circle(center=(90, 50), radius=1.0),
    ],
    solid_road_borders=True,
    # 0.1second = 100ms per step
    dt=0.1
)

In [ ]:
env.world_size

In [ ]:
controller = clothoids.ClothoidTentaclesController(
    num_tentacles=41,
    # t0=7.0,
    t0=10.0,
    # l0=5.0,
    l0=10.0,
    num_points_per_tentacle=64,

    wheelbase=env.WHEELBASE,
    vehicle_width=env.CAR_WIDTH,
    max_lateral_acceleration=4.0,
    max_deceleration=1.5,

    # clearance, curvature, trajectory
    weights=(0.1, 0.2, 0.5),

    # TODO: I set a target velocity of 3.5 but the controller always goes at 5.0 m/s
    target_velocity=3.5,
    kp_velocity=2.0,
)

In [ ]:
obs, info = env.reset()

for i in range(500):
    action = controller.get_action(
        observation=obs,
        path=env.global_path,
        obstacles=env.obstacles,
        road_network=env.road_network,
        max_steering=env.MAX_STEERING,
        max_acceleration=env.MAX_ACCELERATION,
    )
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    env.overlay_manager.clear()
    controller.draw_debug(env, obs, env.global_path)
    
    helpers.preview(env)

    if terminated or truncated:
        break

env.close()

In [ ]:
# Get episode data and compute metrics
episode_data = env.get_episode_data()
# print(f"Episode finished after {i+1} steps")

# Compute metrics
print("\n=== Performance Metrics ===")

cte_metrics = metrics.compute_cross_track_error(
    positions=episode_data['positions'],
    reference_path=env.global_path,
)
print(f"CTE RMS: {cte_metrics['cte_rms']:.3f} m")

smoothness_metrics = metrics.compute_steering_smoothness(
    steering_angles=episode_data['steering_angles'],
    dt=env.dt,
)
print(f"Steering Jerk RMS: {smoothness_metrics['steering_jerk_rms']:.3f} rad/s³")

success = metrics.compute_success_rate([episode_data])
print(f"Success: {'Yes' if success == 1.0 else 'No'}")

In [ ]:
# Create DQN controller and train
dqn_controller = dqn.DQNController(
    n_steering=5,           # 5 steering bins: [-45°, -22.5°, 0°, 22.5°, 45°]
    n_accel=3,              # 3 accel bins: [decel, coast, accel]
    accel_values=(0.3, 0.5, 0.7),
    target_velocity=5.0,
    # DQN hyperparameters
    learning_rate=1e-4,
    buffer_size=50_000,
    learning_starts=500,
    batch_size=64,
    gamma=0.99,
    exploration_fraction=0.3,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
)

# Train on the environment
dqn_controller = dqn_controller.train(
    env=env,
    total_timesteps=20_000,  # Start with 20k for quick test, increase for better results
    log_interval=10,
    progress_bar=True,
    verbose=1,
)

In [ ]:
# Save the trained model
dqn_controller.save("models/dqn_lane_following")

In [ ]:
# Test the trained DQN controller (same interface as other controllers)
obs, info = env.reset()

for i in range(250):
    action = dqn_controller.get_action(
        observation=obs,
        path=env.global_path,
        obstacles=env.obstacles,
        road_network=env.road_network,
        max_steering=env.max_steering,
        max_acceleration=env.max_acceleration,
    )
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    env.overlay_manager.clear()
    dqn_controller.draw_debug(env, obs, env.global_path)
    
    helpers.preview(env)

    if terminated or truncated:
        break

env.close()
print(f"Episode finished after {i+1} steps")

# Get episode data from environment
episode_data = env.get_episode_data()

# Compute metrics
print("\n=== Performance Metrics ===")

# Cross-Track Error
cte_metrics = metrics.compute_cross_track_error(
    positions=episode_data['positions'],
    reference_path=env.global_path,
)
print(f"CTE RMS: {cte_metrics['cte_rms']:.3f} m")
print(f"CTE Mean: {cte_metrics['cte_mean']:.3f} m")
print(f"CTE Max: {cte_metrics['cte_max']:.3f} m")

# Steering Smoothness
smoothness_metrics = metrics.compute_steering_smoothness(
    steering_angles=episode_data['steering_angles'],
    dt=env.dt,
)
print(f"\nSteering Jerk RMS: {smoothness_metrics['steering_jerk_rms']:.3f} rad/s³")
print(f"Steering Jerk Mean: {smoothness_metrics['steering_jerk_mean']:.3f} rad/s³")
print(f"Steering Jerk Max: {smoothness_metrics['steering_jerk_max']:.3f} rad/s³")

# Success Rate (for single episode)
success = metrics.compute_success_rate([episode_data])
print(f"\nSuccess: {'Yes' if success == 1.0 else 'No'} ({success:.1%})")

In [ ]:
import numpy as np
from environments.bicycle.components.roads import (
    RoadBuilder,
    RoadNetwork,
    DOUBLE_LANE,
)

def create_t_junction():
    """
    Create a T-shaped road network:
    
        |     |      |
        |     |      |
    -----------
    
    With vertical roads (left, center, right) and a horizontal road.
    """
    network = RoadNetwork()
    
    # Horizontal road (the base of the T)
    horizontal_road = (
        RoadBuilder((10.0, 50.0), 0.0, DOUBLE_LANE)
        .straight(80.0)
        .build()
    )
    network.add_road(horizontal_road)
    
    # Left vertical road
    left_vertical = (
        RoadBuilder((20.0, 100.0), -np.pi / 2, DOUBLE_LANE)
        .straight(50.0)
        .build()
    )
    network.add_road(left_vertical)
    
    # Center vertical road (connects to middle of horizontal)
    center_vertical = (
        RoadBuilder((50.0, 100.0), -np.pi / 2, DOUBLE_LANE)
        .straight(50.0)
        .build()
    )
    network.add_road(center_vertical)
    
    # Right vertical road
    right_vertical = (
        RoadBuilder((80.0, 100.0), -np.pi / 2, DOUBLE_LANE)
        .straight(50.0)
        .build()
    )
    network.add_road(right_vertical)
    
    return network

# Usage in environment
env = bicycle.BicycleCarEnv(
    road_network=create_t_junction(),
    render_mode="rgb_array",
    spawn=((50.0, 55.0), 0.0),  # Start on horizontal road
    goal=((20.0, 80.0), 2.0),   # Goal on left vertical road
)

In [ ]:
helpers.preview(env)